# Step 4: Provider-Level Feature Engineering

Goal: aggregate the cleaned claim-level table up to **one row per provider**, engineer features that capture billing behavior, claim complexity, and beneficiary profile, then formally test which features are significant and check for multicollinearity before locking in a feature set for modeling.

Two aggregation-design decisions matter throughout this notebook:
1. **Claim-level features** (volume, amounts, code complexity) are aggregated directly from the claim-level table.
2. **Beneficiary demographic features** (age, chronic conditions) are aggregated from a table deduplicated to one row per (Provider, BeneID) first — otherwise a beneficiary with many visits gets their demographics counted multiple times, silently biasing the provider's profile toward whichever beneficiary visits most.

No feature uses `PotentialFraud` as an input anywhere — only as the final label, to avoid leaking target information into the feature set.

In [1]:
import sys
sys.path.append('..')

from src.data_loader import build_claim_level_dataset
from src.cleaning import clean_claim_level_dataset
from src.features import build_provider_features
from src.feature_selection import evaluate_feature_significance, flag_weak_features, compute_vif, iterative_vif_selection

df = clean_claim_level_dataset(build_claim_level_dataset())
provider_features = build_provider_features(df)
provider_features.shape

(5410, 28)

**What this cell does:** runs the full feature engineering pipeline from `src/features.py`, combining ten feature blocks — the original six (claim volume, claim amounts, diagnosis/procedure complexity, physician involvement, beneficiary demographics, claim/stay duration) plus four candidate blocks added after testing them in `03_eda_stats.ipynb`'s "Extended Candidate Features" section (code concentration, physician network spread, beneficiary concentration, per-provider Benford's Law) — into one provider-level table.

**Expected output:** `(5410, 28)` — one row per provider, `Provider` + `PotentialFraud` + 26 engineered features (18 original + 8 new: `TopDiagnosisCodeShare`, `TopProcedureCodeShare`, `TopDiagnosisCodeShareMissing`, `TopProcedureCodeShareMissing`, `AvgPhysicianProviderSpread`, `TopBeneficiaryClaimShare`, `BenfordMAD`, `BenfordMADMissing`).

In [2]:
provider_features.isna().sum()[provider_features.isna().sum() > 0]

Series([], dtype: int64)

**What this checks:** confirms no unexpected missing values slipped through the aggregation. If anything shows up here, it likely means a provider had zero claims of a certain type — several blocks already handle this explicitly rather than leaving NaN: `AvgLengthOfStayDays` fills 0 for providers with no inpatient claims (inside `build_duration_features()`), while `TopDiagnosisCodeShare`/`TopProcedureCodeShare`/`BenfordMAD` fill the *mean* among providers who do have a value — not 0, since a 0-fill would misrepresent "no data" as "no concentration" — each paired with its own `*Missing` flag (`TopDiagnosisCodeShareMissing`, `TopProcedureCodeShareMissing`, `BenfordMADMissing`) so the imputation stays traceable.

In [3]:
provider_features.to_csv('../data/processed/provider_features.csv', index=False)

**What this does:** saves the full provider-level feature table so Step 5 (train/test split) can load it directly. Not committed to git — regenerate locally by re-running this notebook.

## Feature Significance

Before finalizing the feature set, check which features actually separate Fraud from Non-Fraud providers — both statistically (p-value) and practically (effect size). At n=5,410, p-values alone can make almost any feature look "significant," so effect size (Cohen's d) is what actually distinguishes a strong feature from noise.

In [4]:
feature_cols = [c for c in provider_features.columns if c not in ('Provider', 'PotentialFraud')]

sig_df = evaluate_feature_significance(provider_features, feature_cols)
sig_df

,feature,mean_fraud,mean_non_fraud,t_statistic,p_value,cohens_d
0,MaxClaimAmt,43447.332016,9874.863377,29.143041,4.949565e-113,1.605940
1,TopProcedureCodeShareMissing,0.102767,0.679853,-38.304469,5.052946e-181,-1.465599
2,AvgLengthOfStayDays,5.316086,1.868263,28.314255,8.079945e-117,1.205726
3,StdClaimAmt,6376.770794,2016.201421,21.338225,1.786102e-74,1.110047
4,TopProcedureCodeShare,0.205799,0.397075,-20.176438,1.625420e-69,-0.965206
5,BenfordMADMissing,0.069170,0.419046,-26.287242,3.716642e-115,-0.891553
6,TopBeneficiaryClaimShare,0.048289,0.174939,-27.446474,4.420957e-139,-0.747386
7,PropInpatient,0.329196,0.125518,13.792328,1.108756e-37,0.679530
8,NumClaims,420.545455,70.435359,10.879004,6.352459e-25,0.674430
9,NumUniqueOperatingPhysicians,19.286561,5.494902,11.086022,9.296198e-26,0.670120


**How to read this:** sorted by `|Cohen's d|`, largest effect first. The ranking shifted once the 8 new candidate columns joined the table — `MaxClaimAmt` (1.61) is still the single strongest feature, but `TopProcedureCodeShareMissing` (-1.47), `AvgLengthOfStayDays` (1.21), `StdClaimAmt` (1.11), `TopProcedureCodeShare` (-0.97), `BenfordMADMissing` (-0.89), and `TopBeneficiaryClaimShare` (-0.75) now all outrank the *previous* top signal (`NumClaims`, now mid-table at 0.67).

Note the sign: the negative-`d` features mean fraud providers score *lower*, not higher — they use more diverse diagnosis/procedure coding and depend less on a single repeat patient, the reversed-hypothesis finding from Step 3's extended EDA. The two `*Missing` flags ranking this high is itself informative but likely not independent news — both are plausible proxies for claim volume/inpatient share (`NumClaims`, `PropInpatient`), which is exactly what the VIF check below exists to catch. Features near the bottom (small `|Cohen's d|`, high p-value) are candidates for the `flag_weak_features()` check below.

In [5]:
flagged = flag_weak_features(sig_df)
flagged[flagged['is_weak']][['feature', 'p_value', 'cohens_d']]

,feature,p_value,cohens_d
21,AvgNumPhysiciansInvolved,0.000002,0.156819
22,PropDeceasedBeneficiaries,0.074594,0.053463
23,TopDiagnosisCodeShareMissing,0.025332,-0.045175
24,DeductibleToReimbursementRatio,0.175809,-0.027343
25,AvgAgeOfBeneficiaries,0.925385,0.002888


**Important caveat:** a feature flagged "weak" here isn't automatically useless — this test only detects **linear, mean-difference** signal. A feature could still matter to a tree-based model (Random Forest, XGBoost) through nonlinear splits or interactions a t-test can't see. Treat this as a first-pass filter to discuss and justify, not an automatic drop list — for the project report, being able to say "we tested X and found it statistically weak on its own" is itself a useful, defensible finding.

In this run, the weak list is `AvgNumPhysiciansInvolved`, `PropDeceasedBeneficiaries`, `TopDiagnosisCodeShareMissing` (only 5 providers have zero diagnosis-coded claims, so this flag has almost no variance to work with), `DeductibleToReimbursementRatio`, and `AvgAgeOfBeneficiaries`.

## Multicollinearity Check (Variance Inflation Factor)

VIF matters mainly for linear models like logistic regression, which assume independent predictors — highly correlated features destabilize their coefficients. Tree-based models are largely unaffected by multicollinearity, so this check specifically informs which feature set to use with which model.

In [6]:
vif_df = compute_vif(provider_features, feature_cols)
vif_df

,feature,VIF
0,AvgClaimAmt,35.931810
1,MedianClaimAmt,19.465436
2,TopBeneficiaryClaimShare,14.128874
3,TopDiagnosisCodeShare,13.290290
4,PropInpatient,12.246239
5,NumUniqueOperatingPhysicians,10.340936
6,NumUniqueAttendingPhysicians,9.511988
7,StdClaimAmt,9.055033
8,AvgNumDiagnosisCodes,6.943281
9,MaxClaimAmt,5.770656


**What this shows:** the remaining VIF values (after earlier fixes removed exact arithmetic identities like `TotalClaimAmt = NumClaims × AvgClaimAmt`) reflect *genuine* domain correlation, not construction errors — e.g. `AvgClaimAmt`, `MedianClaimAmt`, `StdClaimAmt`, and `MaxClaimAmt` are four different summary statistics of the same claim-amount distribution, so they naturally move together. `PropInpatient` correlates with claim amount because inpatient claims cost more (the same 0.84 correlation found in Step 3's EDA).

Two of the new candidates also show meaningful VIF: `TopBeneficiaryClaimShare` (14.1) and `TopDiagnosisCodeShare` (13.3) — both correlate with the provider-size features above them, since larger/higher-volume providers naturally have more claims to spread across diagnosis codes and more patients to spread claims across. None of this is wrong — it just means logistic regression needs a decorrelated subset.

In [7]:
logistic_features, elimination_log = iterative_vif_selection(provider_features, feature_cols, threshold=10)

print("Final features for logistic regression:")
print(logistic_features)
print("\nElimination log:")
elimination_log

Final features for logistic regression:
['NumClaims', 'MedianClaimAmt', 'StdClaimAmt', 'MaxClaimAmt', 'DeductibleToReimbursementRatio', 'AvgNumDiagnosisCodes', 'AvgNumProcedureCodes', 'AvgNumPhysiciansInvolved', 'NumUniqueAttendingPhysicians', 'AvgAgeOfBeneficiaries', 'PropDeceasedBeneficiaries', 'AvgChronicConditions', 'AvgClaimsPerBeneficiary', 'AvgClaimDurationDays', 'AvgLengthOfStayDays', 'TopDiagnosisCodeShare', 'TopProcedureCodeShare', 'TopDiagnosisCodeShareMissing', 'TopProcedureCodeShareMissing', 'AvgPhysicianProviderSpread', 'BenfordMAD', 'BenfordMADMissing']

Elimination log:


,dropped_feature,vif_at_drop
0,AvgClaimAmt,35.931810
1,TopBeneficiaryClaimShare,14.114356
2,PropInpatient,12.160927
3,NumUniqueOperatingPhysicians,10.301375


**What this does:** greedily drops the single highest-VIF feature, recomputes, and repeats until every remaining feature is below the threshold (10). The `elimination_log` records what was dropped and why — useful evidence for the report that feature selection was deliberate, not arbitrary.

**Result:** `AvgClaimAmt` dropped first (highest VIF, but its information isn't lost — `MedianClaimAmt`/`StdClaimAmt`/`MaxClaimAmt` remain), then `TopBeneficiaryClaimShare` (redundant with the provider-size features once `AvgClaimAmt` is gone), then `PropInpatient` (the known 0.84 correlation with claim amount), then `NumUniqueOperatingPhysicians` (redundant with `NumUniqueAttendingPhysicians`). `TopDiagnosisCodeShare` survives despite a pre-elimination VIF of 13.3 — its correlation was spread across several of the dropped features rather than concentrated in one, so removing them brought it back under threshold without needing to drop it directly.

## Two Feature Sets Going Forward

This is a deliberate modeling choice, not a compromise — different algorithms have different assumptions about their inputs:

| Model type | Feature set | Why |
|---|---|---|
| Logistic Regression | `logistic_features` (22 features, VIF < 10) | Assumes independent predictors; multicollinearity destabilizes coefficients |
| Random Forest / Boosting | `feature_cols` (full 26 features) | Not sensitive to multicollinearity; dropping informative features unnecessarily could lose signal |

Both feature sets are saved below for use in Step 5 onward.

In [8]:
import json

feature_sets = {
    'full_features': feature_cols,
    'logistic_features': logistic_features,
}

with open('../data/processed/feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=2)

feature_sets

{'full_features': ['NumClaims',
  'PropInpatient',
  'AvgClaimAmt',
  'MedianClaimAmt',
  'StdClaimAmt',
  'MaxClaimAmt',
  'DeductibleToReimbursementRatio',
  'AvgNumDiagnosisCodes',
  'AvgNumProcedureCodes',
  'AvgNumPhysiciansInvolved',
  'NumUniqueAttendingPhysicians',
  'NumUniqueOperatingPhysicians',
  'AvgAgeOfBeneficiaries',
  'PropDeceasedBeneficiaries',
  'AvgChronicConditions',
  'AvgClaimsPerBeneficiary',
  'AvgClaimDurationDays',
  'AvgLengthOfStayDays',
  'TopDiagnosisCodeShare',
  'TopProcedureCodeShare',
  'TopDiagnosisCodeShareMissing',
  'TopProcedureCodeShareMissing',
  'AvgPhysicianProviderSpread',
  'TopBeneficiaryClaimShare',
  'BenfordMAD',
  'BenfordMADMissing'],
 'logistic_features': ['NumClaims',
  'MedianClaimAmt',
  'StdClaimAmt',
  'MaxClaimAmt',
  'DeductibleToReimbursementRatio',
  'AvgNumDiagnosisCodes',
  'AvgNumProcedureCodes',
  'AvgNumPhysiciansInvolved',
  'NumUniqueAttendingPhysicians',
  'AvgAgeOfBeneficiaries',
  'PropDeceasedBeneficiaries',
  'A

**What this does:** saves both feature lists to `data/processed/feature_sets.json`, so Step 5 (train/test split) and Step 6 (modeling) can load the appropriate list by name instead of hardcoding column names in multiple places.

## Summary of Step 4

- Built a **provider-level feature table**: 5,410 rows × 26 engineered features across 10 categories — the original 6 (claim volume, claim amounts, diagnosis/procedure complexity, physician involvement, beneficiary demographics, claim/stay duration) plus 4 new candidate blocks (code concentration, physician network spread, beneficiary concentration, per-provider Benford's Law) added after testing them in `03_eda_stats.ipynb`'s Extended Candidate Features section.
- **No target leakage:** no feature uses `PotentialFraud` as an input.
- **Significance testing:** ranked features by effect size (Cohen's d), not p-value alone, since p-values are near-automatically significant at n=5,410.
- **New top signals:** `MaxClaimAmt` remains the single strongest feature (d=1.61), but 6 of the new candidates now outrank the previous top signal (`NumClaims`) — most notably `TopProcedureCodeShareMissing` (d=-1.47) and `TopProcedureCodeShare` (d=-0.97). The negative sign on several new features reflects a genuinely counter-intuitive finding from Step 3: fraud providers show *more* diverse diagnosis/procedure coding and *less* dependence on a single repeat patient, not more repetitive "templated" billing as originally hypothesized.
- **A caught bug worth noting:** the first version of `TopProcedureCodeShare` filled missing values (62.6% of providers have zero procedure-coded claims) with 0, which silently flipped its effect size from d=-0.89 to d=+0.11 — 0 means "many different codes, none dominant," the opposite of "no codes recorded." Fixed by filling with the mean among providers who do have a value, plus a `*Missing` flag for traceability, which restored (and slightly strengthened) the original signal.
- **Multicollinearity resolved in two passes:**
  1. Removed exact arithmetic identities (e.g. `TotalClaimAmt = NumClaims × AvgClaimAmt`, `NumInpatientClaims`/`NumOutpatientClaims` vs `NumClaims`/`PropInpatient`) directly in `features.py`, so they can't resurface on re-runs.
  2. Used iterative VIF elimination for genuine (non-identity) correlations — this time dropping 4 features (`AvgClaimAmt`, `TopBeneficiaryClaimShare`, `PropInpatient`, `NumUniqueOperatingPhysicians`) instead of the previous 3, producing a 22-feature decorrelated set for logistic regression.
- **Two feature sets saved** (`feature_sets.json`): full 26-feature set for tree-based/boosting models, 22-feature VIF-clean subset for logistic regression.
- **Not yet resolved:** `TopProcedureCodeShareMissing` and `BenfordMADMissing` both rank as strong standalone signals but are plausible proxies for claim volume/inpatient share already captured elsewhere. The VIF check only flags *linear* correlation and didn't eliminate either (VIF 5.1 and 2.1 respectively) — worth a closer look at their correlation with `NumClaims`/`PropInpatient` specifically before trusting them as independent evidence in the final report.